# Host–Pathogen Arms Race as an Adversarial Game

In this notebook we will explore a **toy model of the evolutionary arms race** between a **host (or clinician)** and a **pathogen**.  
The aim is to illustrate how concepts from **adversarial AI and decision-making** can be applied to **bioinformatics and systems biology**.

---

## 📌 Problem Setup

- **Pathogen genotype:**  
  Represented as an 8-bit binary string (`00000000`).  
  - Each bit corresponds to the presence (1) or absence (0) of a resistance mutation.  
  - Mutations can make the pathogen resistant to certain drugs, but each resistance bit carries a **fitness cost**.

- **Host (Leader):**  
  Chooses a **treatment policy** at the beginning and sticks with it for the whole simulation.  
  Available policies:  
  - `DrugA`  
  - `DrugB`  
  - `Combo` (A + B together)  
  - `Holiday` (no treatment)

- **Pathogen (Follower):**  
  Adapts stochastically over time by:  
  - **Mutation** (each bit may flip with probability μ per replication cycle).  
  - **Selection** (mutants with higher fitness under the current treatment are more likely to dominate).  

- **Burden dynamics:**  
  The pathogen population grows (baseline growth rate), reduced by drug kill effects, but mutations reduce susceptibility.  
  Each resistance bit reduces susceptibility to a drug but imposes a **cost to growth**.  

---

## 🎯 Objective

- The **host’s goal**: choose a policy that **minimizes the probability of pathogen escape**,  
  where *escape* is defined as the pathogen burden exceeding a threshold (e.g., 10× the initial load) within a fixed number of replication cycles.  
- The **pathogen’s goal**: survive and expand despite treatment.  
- This creates a **leader–follower (Stackelberg) game**:  
  - The host commits to a policy.  
  - The pathogen then responds through mutation and selection dynamics.  

---

## 🔍 What You Will See

- How different treatment policies affect the **evolution of pathogen genotypes** over time.  
- Traces showing:  
  - Pathogen genome (8 bits)  
  - Pathogen burden  
  - Host policy applied  
- A **Monte Carlo simulation** estimating the probability of escape under each host policy.  

---

➡️ This simplified model lets us **connect adversarial reasoning from AI** (host vs pathogen) with **bioinformatics concepts** such as mutation, resistance, and therapy design.  
It also illustrates why *robust treatment policies* are needed: the pathogen acts like an **adaptive opponent**.  


In [2]:
# Re-run after state reset (same code, compacted execution)

import random, math
random.seed(7)

N_BITS = 8
MU = 0.02
BETA = 4.0
COST_PER_MUT = 0.04
BASE_GROWTH = 1.10
KILL_A = 0.55
KILL_B = 0.55
KILL_COMBO = 0.85
T_STEPS = 40
BURDEN0 = 1.0
ESCAPE_THRESHOLD = 10.0
RUNS = 400

RES_BITS_A = {0, 3, 5}
RES_BITS_B = {1, 2, 6}

def int_to_bits(x, n=N_BITS): return [(x >> i) & 1 for i in range(n)]
def bits_to_int(bits):
    v = 0
    for i, b in enumerate(bits): v |= (b & 1) << i
    return v
def mutate_bits(bits, mu=MU): return [b ^ 1 if random.random() < mu else b for b in bits]
def resistance_fraction(bits, res_set):
    if not res_set: return 0.0
    return sum(bits[i] for i in res_set) / len(res_set)

def fitness(bits, treatment):
    ones = sum(bits)
    cost = COST_PER_MUT * ones
    if treatment == "A":
        resist = resistance_fraction(bits, RES_BITS_A); kill = KILL_A * (1.0 - resist)
    elif treatment == "B":
        resist = resistance_fraction(bits, RES_BITS_B); kill = KILL_B * (1.0 - resist)
    elif treatment == "Combo":
        rA = resistance_fraction(bits, RES_BITS_A); rB = resistance_fraction(bits, RES_BITS_B)
        kill = KILL_COMBO * (1.0 - 0.5*(rA+rB))
    elif treatment == "Holiday":
        kill = 0.0
    else: raise ValueError
    g = BASE_GROWTH * (1.0 - kill) - cost
    return g

def step_dynamics(bits, burden, treatment):
    cand = mutate_bits(bits, mu=MU)
    f_curr, f_cand = fitness(bits, treatment), fitness(cand, treatment)
    p_cand = math.exp(4.0 * f_cand) / (math.exp(4.0 * f_cand) + math.exp(4.0 * f_curr))
    next_bits = cand if random.random() < p_cand else bits
    g = fitness(next_bits, treatment)
    next_burden = burden * max(0.0, 1.0 + g - 1.0)
    return next_bits, next_burden

def simulate(policy, genotype0=0, runs=RUNS):
    escapes = 0
    for _ in range(runs):
        bits = int_to_bits(genotype0); burden = BURDEN0
        for _ in range(T_STEPS):
            bits, burden = step_dynamics(bits, burden, policy)
            if burden >= ESCAPE_THRESHOLD:
                escapes += 1; break
    return escapes / runs

# Trace function: print genome (8 bits), burden, and host policy at each iteration.

# We assume the previous cell defined: N_BITS, int_to_bits, step_dynamics, BURDEN0, ESCAPE_THRESHOLD

def format_bits(bits):
    # Show bits from MSB..LSB for readability (bit7 ... bit0)
    return ''.join(str(bits[i]) for i in reversed(range(len(bits))))

def trace_run(policy="A", genotype0=0, steps=20):
    bits = int_to_bits(genotype0)
    burden = BURDEN0
    print(f"Tracing policy='{policy}' for {steps} steps (ESCAPE if burden ≥ {ESCAPE_THRESHOLD})")
    print(f"{'iter':>4s}  {'genome':>10s}  {'burden':>10s}  {'policy':>7s}")
    print("-"*40)
    for t in range(steps):
        print(f"{t:4d}  {format_bits(bits):>10s}  {burden:10.4f}  {policy:>7s}")
        # Advance one step
        bits, burden = step_dynamics(bits, burden, policy)
        if burden >= ESCAPE_THRESHOLD:
            print(f"{t+1:4d}  {format_bits(bits):>10s}  {burden:10.4f}  {policy:>7s}   <-- ESCAPE")
            break

# Example traces
trace_run(policy="A", genotype0=0, steps=20)
print()
trace_run(policy="Holiday", genotype0=0, steps=20)


Tracing policy='A' for 20 steps (ESCAPE if burden ≥ 10.0)
iter      genome      burden   policy
----------------------------------------
   0    00000000      1.0000        A
   1    00000000      0.4950        A
   2    00000000      0.2450        A
   3    00000000      0.1213        A
   4    00000000      0.0600        A
   5    00000000      0.0297        A
   6    00000000      0.0147        A
   7    00000000      0.0073        A
   8    00000000      0.0036        A
   9    00000000      0.0018        A
  10    00000000      0.0009        A
  11    00000000      0.0004        A
  12    10000000      0.0002        A
  13    10000000      0.0001        A
  14    10000000      0.0000        A
  15    10010000      0.0000        A
  16    10010000      0.0000        A
  17    10010000      0.0000        A
  18    10010000      0.0000        A
  19    10010000      0.0000        A

Tracing policy='Holiday' for 20 steps (ESCAPE if burden ≥ 10.0)
iter      genome      burden   policy


In [3]:
# Optimal policy
policies = ["A","B","Combo","Holiday"]
results = {p: simulate(p, genotype0=0, runs=RUNS) for p in policies}
print("Host–Pathogen (Leader–Follower, Stochastic) — Policy Risk Table")
print(f"(N={RUNS} runs, horizon={T_STEPS}, escape if burden ≥ {ESCAPE_THRESHOLD})\n")
for p in policies:
    print(f"  Policy {p:7s}  →  P(escape) ≈ {results[p]:.3f}")
best_policy = min(results, key=results.get)
print(f"\nRecommended (Stackelberg, one-shot): {best_policy}  (minimizes estimated escape risk)\n")

# Sensitivity: start with resistance to A
resA_start = 0
for i in RES_BITS_A: resA_start |= (1 << i)
sens = {p: simulate(p, genotype0=resA_start, runs=RUNS) for p in policies}
print("Sensitivity: start with resistance to A (bits 0,3,5=1)")
for p in policies:
    print(f"  Policy {p:7s}  →  P(escape) ≈ {sens[p]:.3f}")

Host–Pathogen (Leader–Follower, Stochastic) — Policy Risk Table
(N=400 runs, horizon=40, escape if burden ≥ 10.0)

  Policy A        →  P(escape) ≈ 0.000
  Policy B        →  P(escape) ≈ 0.000
  Policy Combo    →  P(escape) ≈ 0.000
  Policy Holiday  →  P(escape) ≈ 0.470

Recommended (Stackelberg, one-shot): A  (minimizes estimated escape risk)

Sensitivity: start with resistance to A (bits 0,3,5=1)
  Policy A        →  P(escape) ≈ 0.000
  Policy B        →  P(escape) ≈ 0.000
  Policy Combo    →  P(escape) ≈ 0.000
  Policy Holiday  →  P(escape) ≈ 0.005


In [8]:
# Extended trace: switch host policy over time.
# - Holiday for 100 steps
# - then B for 100 steps
# - then A for 100 steps

def long_trace(policies, switch_steps, genotype0=0, total_steps=300):
    bits = int_to_bits(genotype0)
    burden = BURDEN0
    print(f"Tracing long run: policies={policies}, switch every {switch_steps} steps")
    print(f"{'iter':>4s}  {'genome':>10s}  {'burden':>10s}  {'policy':>7s}")
    print("-"*50)
    for t in range(total_steps):
        phase = t // switch_steps
        if phase >= len(policies):
            policy = policies[-1]
        else:
            policy = policies[phase]
        print(f"{t:4d}  {format_bits(bits):>10s}  {burden:10.4f}  {policy:>7s}")
        bits, burden = step_dynamics(bits, burden, policy)
        if burden >= ESCAPE_THRESHOLD:
            print(f"{t+1:4d}  {format_bits(bits):>10s}  {burden:10.4f}  {policy:>7s}   <-- ESCAPE")
            break

# Run the requested trace: 100 steps Holiday, 100 steps B, 100 steps A
long_trace(["Holiday","B","Holiday"], switch_steps=30, genotype0=0, total_steps=150)


Tracing long run: policies=['Holiday', 'B', 'Holiday'], switch every 30 steps
iter      genome      burden   policy
--------------------------------------------------
   0    00000000      1.0000  Holiday
   1    00000000      1.1000  Holiday
   2    00000000      1.2100  Holiday
   3    00000000      1.3310  Holiday
   4    00000000      1.4641  Holiday
   5    00000000      1.6105  Holiday
   6    00000000      1.7716  Holiday
   7    00000000      1.9487  Holiday
   8    00000000      2.1436  Holiday
   9    00000000      2.3579  Holiday
  10    00000000      2.5937  Holiday
  11    00000000      2.8531  Holiday
  12    00000000      3.1384  Holiday
  13    00000000      3.4523  Holiday
  14    00000000      3.7975  Holiday
  15    00000000      4.1772  Holiday
  16    01000000      4.4279  Holiday
  17    01000000      4.6936  Holiday
  18    01000000      4.9752  Holiday
  19    01000000      5.2737  Holiday
  20    11000000      5.3792  Holiday
  21    11000000      5.4867  Holid

In [12]:
long_trace(["Holiday"], switch_steps=30, genotype0=resA_start, total_steps=100)


Tracing long run: policies=['Holiday'], switch every 30 steps
iter      genome      burden   policy
--------------------------------------------------
   0    00101001      1.0000  Holiday
   1    10100001      0.9800  Holiday
   2    10100001      0.9604  Holiday
   3    10100001      0.9412  Holiday
   4    10100001      0.9224  Holiday
   5    10100001      0.9039  Holiday
   6    10100001      0.8858  Holiday
   7    10100001      0.8681  Holiday
   8    10100001      0.8508  Holiday
   9    10100001      0.8337  Holiday
  10    10100001      0.8171  Holiday
  11    10100001      0.8007  Holiday
  12    10100001      0.7847  Holiday
  13    10100001      0.7690  Holiday
  14    10100001      0.7536  Holiday
  15    10100001      0.7386  Holiday
  16    11100001      0.6943  Holiday
  17    11100001      0.6526  Holiday
  18    11100001      0.6134  Holiday
  19    11100001      0.5766  Holiday
  20    11100001      0.5420  Holiday
  21    11100001      0.5095  Holiday
  22    11100

In [16]:
# Now cost per mutation is cheaper
COST_PER_MUT = 0.02
long_trace(["A"], switch_steps=30, genotype0=resA_start, total_steps=100)


Tracing long run: policies=['A'], switch every 30 steps
iter      genome      burden   policy
--------------------------------------------------
   0    00101001      1.0000        A
   1    00101001      1.0400        A
   2    00101001      1.0816        A
   3    00101001      1.1249        A
   4    00101001      1.1699        A
   5    00101001      1.2167        A
   6    00101001      1.2653        A
   7    00101001      1.3159        A
   8    00101000      1.1295        A
   9    00101000      0.9695        A
  10    00101000      0.8321        A
  11    00101000      0.7143        A
  12    00101010      0.5988        A
  13    00101010      0.5020        A
  14    00101010      0.4208        A
  15    00101010      0.3528        A
  16    00101010      0.2958        A
  17    00101010      0.2479        A
  18    00101010      0.2079        A
  19    00101010      0.1743        A
  20    00101010      0.1461        A
  21    00101010      0.1225        A
  22    00101010   